In [1]:
from pathlib import Path

import pandas as pd

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent

processed = root / "data" / "processed"
case_ids = [695464, 3123427, 3863638]
case_filter = [("source_row", "in", case_ids)]

transactions = pd.read_parquet(
    processed / "lab02_validation.parquet",
    columns=["source_row", "timestamp", "payment_format", "amount_paid", "payment_currency"],
    filters=case_filter,
)

history = pd.read_parquet(
    processed / "lab03_history_features.parquet",
    columns=[
        "source_row",
        "sender_in_count_24h",
        "sender_out_count_24h",
        "receiver_in_count_24h",
        "receiver_out_count_24h",
    ],
    filters=case_filter,
)

cases = (
    transactions.merge(history, on="source_row", validate="one_to_one")
    .set_index("source_row")
    .loc[case_ids]
)

display(cases)

,timestamp,payment_format,amount_paid,payment_currency,sender_in_count_24h,sender_out_count_24h,receiver_in_count_24h,receiver_out_count_24h
source_row,,,,,,,,
695464,2022-09-07 11:09:00,ACH,18533.55,US Dollar,3,2,0,0
3123427,2022-09-07 17:29:00,ACH,1003214.81,Rupee,17,18,0,0
3863638,2022-09-08 13:50:00,ACH,13986.49,Euro,1,5,0,0


### Reference Questions:
695464: The sender received three payments → How soon before this ACH did each one arrive?

3123427: The sender received 17 payments and sent 18 in the prior 24 hours → How did the total outgoing amount compare with the total incoming amount during that window?

3863638: The receiver had no recorded incoming or outgoing payments in the prior 24 hours → When was its most recent transaction before that window?

# Step 2 — Retrieve the evidence for each alert

In [2]:
def describe_case(row):
    return (
        f"The sender made an outgoing {row.payment_format} payment of "
        f"{row.amount_paid:,.2f} {row.payment_currency}. "
        f"In the preceding 24 hours, the sender received {row.sender_in_count_24h} "
        f"and sent {row.sender_out_count_24h} payments. "
        f"In that same period, the receiver received {row.receiver_in_count_24h} "
        f"and sent {row.receiver_out_count_24h} payments."
    )


cases["alert_text"] = [describe_case(row) for row in cases.itertuples()]
print(cases.at[3863638, "alert_text"])

The sender made an outgoing ACH payment of 13,986.49 Euro. In the preceding 24 hours, the sender received 1 and sent 5 payments. In that same period, the receiver received 0 and sent 0 payments.


In [4]:
import torch
from sentence_transformers import SentenceTransformer

passages = pd.read_parquet(
    processed / "lab12" / "guidance_passages.parquet"
).set_index("passage_id")

torch.set_num_threads(2)
encoder = SentenceTransformer(
    str(root / "models" / "lab11_original_encoder"),
    device="cpu",
    local_files_only=True,
)

passage_vectors = encoder.encode(passages["text"].tolist(), normalize_embeddings=True)
alert_vectors = encoder.encode(cases["alert_text"].tolist(), normalize_embeddings=True)

scores = pd.DataFrame(
    alert_vectors @ passage_vectors.T,
    index=cases.index,
    columns=passages.index,
)

contexts = {}
for alert_id, passage_scores in scores.iterrows():
    top_ids = passage_scores.nlargest(2).index
    lines = [f"[TX-{alert_id}] {cases.at[alert_id, 'alert_text']}"]
    lines.extend(f"[{pid}] {passages.at[pid, 'text']}" for pid in top_ids)
    contexts[alert_id] = "\n".join(lines)

    print(f"{alert_id}: {', '.join(top_ids)}")

/Users/sara/Projects/data-science-lab/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1126 16:21:25.544000 98146 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4236.96it/s]


695464: P01, P06
3123427: P03, P01
3863638: P06, P03


In [10]:
print(contexts[3863638])

[TX-3863638] The sender made an outgoing ACH payment of 13,986.49 Euro. In the preceding 24 hours, the sender received 1 and sent 5 payments. In that same period, the receiver received 0 and sent 0 payments.
[P06] The size or type of a payment differs from what is expected for the customer.
[P03] Funds enter and leave the same account over a short interval.


# Step 3 — Generate six notes from the same evidence

In [11]:
def make_prompt(context):
    return f"""Write a two-sentence transaction review note using the evidence below.

Observation: State what the transaction record shows and cite its [TX-...] ID.
Follow-up: Suggest one specific check that would give the investigator new information.
Cite a [P...] passage if it supports the check. If neither passage supports it,
say so briefly. Treat guidance as a possible lead, not proof of what happened.

Return JSON with the fields observation and follow_up.

Evidence:
{context}"""


note_schema = {
    "type": "object",
    "properties": {
        "observation": {"type": "string"},
        "follow_up": {"type": "string"},
    },
    "required": ["observation", "follow_up"],
    "additionalProperties": False,
}

In [ ]:
import json
from urllib.request import Request, urlopen

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(root / ".env")
client = OpenAI()


def qwen_note(prompt):
    payload = {
        "model": "qwen3:4b",
        "prompt": prompt,
        "format": note_schema,
        "stream": False,
        "think": False,
        "options": {"temperature": 0, "num_ctx": 2048, "num_predict": 128},
        "keep_alive": 0,
    }
    request = Request(
        "http://localhost:11434/api/generate",
        data=json.dumps(payload).encode(),
        headers={"Content-Type": "application/json"},
    )
    with urlopen(request, timeout=180) as reply:
        result = json.load(reply)
    return json.loads(result["response"])


def luna_note(prompt):
    response = client.responses.create(
        model="gpt-6-sol",
        reasoning={"effort": "none"},
        input=prompt,
        text={"format": {
            "type": "json_schema",
            "name": "review_note",
            "strict": True,
            "schema": note_schema,
        }},
        max_output_tokens=128,
        store=False,
    )
    if response.status != "completed" or not response.output_text:
        raise RuntimeError(f"No complete note: {response.incomplete_details}")
    return json.loads(response.output_text)


notes = {}

In [ ]:
for alert_id, context in contexts.items():
    prompt = make_prompt(context)

    for model, generate in [("Qwen", qwen_note), ("Sol", luna_note)]:
        key = (alert_id, model)
        if key not in notes:
            print(f"Generating {model} note for {alert_id}...", flush=True)
            notes[key] = generate(prompt)

Generating Qwen note for 695464...
Generating Luna note for 695464...
Generating Qwen note for 3123427...
Generating Luna note for 3123427...
Generating Qwen note for 3863638...
Generating Luna note for 3863638...


In [15]:
notes

{(695464,
  'Qwen'): {'observation': 'TX-695464 shows the sender made an outgoing ACH payment of $18,533.55, with 3 incoming and 2 outgoing payments in the prior 24 hours, while the receiver had no activity.', 'follow_up': "Check if the sender's account has a recent transaction history showing unusual payment patterns to identify potential fraud. [P01] indicates payment volume differs from usual activity."},
 (695464,
  'Luna'): {'observation': '[TX-695464] shows an outgoing ACH payment of $18,533.55; in the preceding 24 hours, the sender received three payments and sent two, while the receiver had no other payment activity.', 'follow_up': "Compare this payment's amount and type with the sender's historical payments to determine whether it is unusual for the customer, as suggested by [P06]."},
 (3123427,
  'Qwen'): {'observation': 'TX-3123427 shows the sender made an outgoing ACH payment of 1,003,214.81 Rupee, with 17 incoming and 18 outgoing payments in the prior 24 hours, while the r

In [ ]:
for model in ["Qwen", "Sol"]:
    note = notes[(3863638, model)]
    print(f"\n{model}\n{note['observation']}\n{note['follow_up']}")


Qwen
TX-3863638 shows the sender made an outgoing ACH payment of 13,986.49 Euro, with 1 incoming and 5 outgoing payments in the prior 24 hours, while the receiver had no activity.
Check if the payment amount aligns with the customer's typical transaction size using P06, which indicates discrepancies in payment size/type may signal unusual activity.

Luna
[TX-3863638] shows an outgoing ACH payment of €13,986.49; in the preceding 24 hours, the sender received one payment and sent five, while the receiver had no recorded payments.
Compare the payment amount and ACH method with the sender’s usual activity to see whether either is unusual for the customer, as suggested by [P06].


#### Both models proposed a supported sender-profile check. The second note more clearly identified the sender and limited the receiver observation to the recorded 24-hour window.”

In [19]:
for alert_id in [695464, 3123427]:
    print(f"\n--- Alert {alert_id} ---")
    print(contexts[alert_id])

    for model in ["Qwen", "Luna"]:
        note = notes[(alert_id, model)]
        print(f"\n{model}")
        print(note["observation"])
        print(note["follow_up"])


--- Alert 695464 ---
[TX-695464] The sender made an outgoing ACH payment of 18,533.55 US Dollar. In the preceding 24 hours, the sender received 3 and sent 2 payments. In that same period, the receiver received 0 and sent 0 payments.
[P01] Payment volume differs from the customer's usual activity.
[P06] The size or type of a payment differs from what is expected for the customer.

Qwen
TX-695464 shows the sender made an outgoing ACH payment of $18,533.55, with 3 incoming and 2 outgoing payments in the prior 24 hours, while the receiver had no activity.
Check if the sender's account has a recent transaction history showing unusual payment patterns to identify potential fraud. [P01] indicates payment volume differs from usual activity.

Luna
[TX-695464] shows an outgoing ACH payment of $18,533.55; in the preceding 24 hours, the sender received three payments and sent two, while the receiver had no other payment activity.
Compare this payment's amount and type with the sender's historical

In [20]:
passages

,source_section,text,source_url
passage_id,,,
P01,Client profile,Payment volume differs from the customer's usu...,https://fintrac-canafe.canada.ca/guidance-dire...
P02,Products and services,A customer overpays a credit card and then req...,https://fintrac-canafe.canada.ca/guidance-dire...
P03,Atypical activity,Funds enter and leave the same account over a ...,https://fintrac-canafe.canada.ca/guidance-dire...
P04,Identification,Identity records supplied by a customer contai...,https://fintrac-canafe.canada.ca/guidance-dire...
P05,Changes in account activity,Payments resume after an account has been dorm...,https://fintrac-canafe.canada.ca/guidance-dire...
P06,Client profile,The size or type of a payment differs from wha...,https://fintrac-canafe.canada.ca/guidance-dire...
P07,Products and services,An account receives money and passes it onward...,https://fintrac-canafe.canada.ca/guidance-dire...
P08,Client behaviour,The customer gives an unclear explanation of a...,https://fintrac-canafe.canada.ca/guidance-dire...
P09,Atypical activity,Transactions repeatedly occur at a particular ...,https://fintrac-canafe.canada.ca/guidance-dire...
